# MediGuide — Baseline Evaluation
### Qwen2.5-1.5B-Instruct on Doctor-Patient Conversations (pre-LoRA baseline)

This notebook establishes the **zero-shot baseline** for the base `Qwen/Qwen2.5-1.5B-Instruct`
model on the doctor-patient conversation dataset, before any LoRA fine-tuning.

**Why a baseline first:** we need a reference point (ROUGE-L, BLEU, perplexity, latency) to
know whether LoRA fine-tuning actually helps, and by how much, later.

**Environment:** Kaggle, single T4 (16GB). T4 = Turing architecture (compute capability 7.5) →
no native bf16 tensor-core support, so we use **fp16** throughout, not bf16.

**Known pitfall we're avoiding this run:** last training run added tokens to the tokenizer and
called `resize_token_embeddings`, which desynced the LoRA adapter's embedding table from the
base model's (`151665` vs `151936` rows) and forced `embed_tokens`/`lm_head` into
`modules_to_save`, bloating the adapter to ~905MB. This time we use the **stock Qwen2.5
tokenizer completely unmodified** — no `add_tokens`, no `resize_token_embeddings`.


## 1. Setup

In [ ]:
!pip install -q -U peft accelerate bitsandbytes evaluate rouge_score sacrebleu sentencepiece


In [ ]:
import os, json, time, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# --- Locate dataset files -----------------------------------------------
# Adjust DATA_DIR if your mounted dataset path differs.
DATA_DIR = "/kaggle/input/datasets/totaldose/doctor-patient-conversation/data/processed"

if os.path.isdir(DATA_DIR):
    print("Files in DATA_DIR:")
    for f in sorted(os.listdir(DATA_DIR)):
        print(" -", f)
else:
    print(f"WARNING: {DATA_DIR} not found in this environment. "
          "Set DATA_DIR to wherever train/val/test.json live before continuing.")


In [ ]:
# Resolve exact filenames (handles val vs valid/dev naming variants)
def find_split_file(data_dir, split_names):
    for name in split_names:
        p = os.path.join(data_dir, f"{name}.json")
        if os.path.exists(p):
            return p
    return None

TRAIN_PATH = find_split_file(DATA_DIR, ["train"])
VAL_PATH   = find_split_file(DATA_DIR, ["val", "valid", "validation", "dev"])
TEST_PATH  = find_split_file(DATA_DIR, ["test"])

print("train:", TRAIN_PATH)
print("val:  ", VAL_PATH)
print("test: ", TEST_PATH)

assert TEST_PATH is not None, "Could not find test.json — check DATA_DIR"


## 2. Tokenizer sanity check

This is the guard against the exact bug from last time: we load the **unmodified** tokenizer,
confirm it already has everything Qwen2.5-Instruct needs (chat tokens, pad token), and print
both the tokenizer length and the model's configured `vocab_size` so any future mismatch is
caught immediately, before training — not after a 900MB adapter is saved.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("len(tokenizer):        ", len(tokenizer))
print("config.vocab_size:     ", config.vocab_size)
print("pad_token:             ", tokenizer.pad_token)
print("eos_token:              ", tokenizer.eos_token)
print("chat_template present: ", tokenizer.chat_template is not None)

# Qwen2.5's config.vocab_size is padded to a multiple for tensor-core efficiency, so it will
# legitimately be >= len(tokenizer). That is NORMAL and not the bug we hit before. The bug only
# happens if we call add_tokens/resize_token_embeddings ourselves. We deliberately do neither.
assert config.vocab_size >= len(tokenizer), "Unexpected: tokenizer is larger than model vocab_size!"

if tokenizer.pad_token is None:
    print("No pad_token set -> reusing eos_token as pad_token (no new token added).")
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"  # required for correct batched generation with decoder-only models


## 3. Load the base model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16
).to(DEVICE)
model.eval()
print("Model loaded on", DEVICE)


## 4. Load and inspect the dataset

In [ ]:
def load_split(path):
    if path is None:
        return None
    with open(path) as f:
        data = json.load(f)
    return pd.DataFrame(data)

df_train = load_split(TRAIN_PATH)
df_val   = load_split(VAL_PATH)
df_test  = load_split(TEST_PATH)

for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    if df is not None:
        print(f"{name}: {len(df)} rows")

df_test.head(3)


In [ ]:
# Severity ('Status') distribution — treated as metadata for stratified reporting only,
# NOT part of the model input/target (the task spec doesn't ask the model to predict severity).
print(df_test['Status'].value_counts())
df_test['Status'].value_counts().plot(kind='bar', title='Test set severity distribution')
plt.ylabel('count')
plt.show()


In [ ]:
# Word-level length stats (quick sanity check)
for col in ['Description', 'Patient', 'Doctor']:
    lens = df_test[col].str.split().apply(len)
    print(f"{col:12s} words -> min {lens.min():4d}  median {lens.median():6.1f}  "
          f"mean {lens.mean():6.1f}  max {lens.max():4d}")


In [ ]:
# Token-level length stats using the ACTUAL tokenizer — this is what determines max_length,
# not the word counts above.
sample_n = min(300, len(df_test))
sample = df_test.sample(sample_n, random_state=SEED)

patient_tok_lens = sample['Patient'].apply(lambda t: len(tokenizer.encode(t)))
doctor_tok_lens  = sample['Doctor'].apply(lambda t: len(tokenizer.encode(t)))

print("Patient tokens -> median", patient_tok_lens.median(), " p95", patient_tok_lens.quantile(0.95))
print("Doctor  tokens -> median", doctor_tok_lens.median(), " p95", doctor_tok_lens.quantile(0.95))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(patient_tok_lens, bins=30); axes[0].set_title('Patient tokens')
axes[1].hist(doctor_tok_lens, bins=30); axes[1].set_title('Doctor tokens')
plt.tight_layout(); plt.show()


## 5. Prompt template

We frame this as a single-turn chat: a **system** message carrying the clinical/professional/
disclaimer requirements from the project spec, a **user** turn combining `Description` +
`Patient` (the patient's framing + their free-text question), and the **assistant** turn is the
`Doctor` response we're training toward / evaluating against.

We use `tokenizer.apply_chat_template` so this matches exactly what Qwen2.5-Instruct expects —
no hand-rolled special tokens.


In [ ]:
SYSTEM_PROMPT = (
    "You are a medical information assistant. Respond to the patient's question with "
    "clear, professional, and clinically sound guidance, consistent with recognized clinical "
    "guidelines. Use formal medical language appropriate for a patient audience. Always make "
    "clear that your response is informational only and does not replace an in-person "
    "diagnosis or professional medical care."
)

def build_prompt(description, patient, for_generation=True):
    user_turn = f"{description.strip()}\n\n{patient.strip()}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_turn},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_generation
    )

# Demo
demo = df_test.iloc[0]
print(build_prompt(demo['Description'], demo['Patient']))


In [ ]:
MAX_NEW_TOKENS = 256          # covers p95 Doctor-response length with margin
MAX_PROMPT_LENGTH = 512        # covers p95 Patient-turn length with margin
GEN_BATCH_SIZE = 8


## 6. Baseline generation (single GPU)

In [ ]:
def generate_batch(prompts, max_new_tokens=MAX_NEW_TOKENS):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy: deterministic, reproducible baseline
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], elapsed / len(prompts)


In [ ]:
prompts = [build_prompt(row["Description"], row["Patient"]) for _, row in df_test.iterrows()]

predictions = []
gen_seconds = []

t0 = time.time()
for i in range(0, len(prompts), GEN_BATCH_SIZE):
    batch = prompts[i:i + GEN_BATCH_SIZE]
    texts, per_ex_time = generate_batch(batch)
    predictions.extend(texts)
    gen_seconds.extend([per_ex_time] * len(texts))
    print(f"{min(i + GEN_BATCH_SIZE, len(prompts))}/{len(prompts)} done, "
          f"{time.time() - t0:.1f}s elapsed", flush=True)

df_test["prediction"] = predictions
df_test["gen_seconds"] = gen_seconds
total_gen_time = time.time() - t0
print(f"Total generation time: {total_gen_time:.1f}s")


In [ ]:
df_test[["Description", "Doctor", "prediction"]].head(3)


## 7. Perplexity (teacher-forced)

Perplexity here is computed by teacher-forcing the base model on `prompt + Doctor` and
measuring loss **only over the `Doctor` response tokens** (prompt tokens masked with `-100`).
Lower is better; this tells us how well the base model's own distribution already fits the
reference doctor responses, independent of generation/sampling choices.


In [ ]:
def compute_example_loss(prompt, target):
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=MAX_PROMPT_LENGTH)["input_ids"][0]
    target_ids = tokenizer(target + tokenizer.eos_token, return_tensors="pt",
                            add_special_tokens=False)["input_ids"][0]
    input_ids = torch.cat([prompt_ids, target_ids]).unsqueeze(0).to(DEVICE)
    labels = input_ids.clone()
    labels[:, :len(prompt_ids)] = -100  # mask prompt, only score the Doctor response

    with torch.no_grad():
        out = model(input_ids=input_ids, labels=labels)
    return out.loss.item(), len(target_ids)


In [ ]:
losses = []
n_target_tokens = []

t0 = time.time()
for i, row in df_test.iterrows():
    prompt = build_prompt(row["Description"], row["Patient"])
    loss, n_tok = compute_example_loss(prompt, row["Doctor"])
    losses.append(loss)
    n_target_tokens.append(n_tok)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(df_test)} scored, {time.time() - t0:.1f}s elapsed", flush=True)

df_test["loss"] = losses
df_test["n_target_tokens"] = n_target_tokens
print(f"PPL scoring time: {time.time() - t0:.1f}s")


In [ ]:
# Corpus-level perplexity: weight each example's loss by its target token count,
# not a plain mean-of-per-example-perplexities (which is a biased estimator).
total_loss_tokens = (df_test["loss"] * df_test["n_target_tokens"]).sum()
total_tokens = df_test["n_target_tokens"].sum()
corpus_ppl = math.exp(total_loss_tokens / total_tokens)
print(f"Corpus-level perplexity (baseline, zero-shot): {corpus_ppl:.3f}")

# Per-example perplexity for stratified breakdowns later
df_test["perplexity"] = df_test["loss"].apply(math.exp)


## 8. ROUGE-L and BLEU

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

references = df_test["Doctor"].tolist()

rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
bleu_scores = bleu.compute(predictions=predictions, references=[[r] for r in references])

print("ROUGE:", rouge_scores)
print("BLEU: ", bleu_scores["score"])


In [ ]:
# Per-example ROUGE-L for stratified analysis
rouge_per_row = []
for pred, ref in zip(predictions, references):
    r = rouge.compute(predictions=[pred], references=[ref], use_stemmer=True)
    rouge_per_row.append(r["rougeL"])
df_test["rougeL"] = rouge_per_row


## 9. Baseline results summary

In [ ]:
avg_gen_seconds = df_test["gen_seconds"].mean()

summary = {
    "model": MODEL_NAME,
    "seed": SEED,
    "n_test_examples": len(df_test),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "bleu": bleu_scores["score"],
    "perplexity_corpus": corpus_ppl,
    "avg_generation_seconds_per_example": avg_gen_seconds,
    "max_new_tokens": MAX_NEW_TOKENS,
    "total_wall_clock_generation_seconds": total_gen_time,
}
pd.DataFrame([summary]).T.rename(columns={0: "value"})


In [ ]:
# Stratified by severity — does the zero-shot base model already do better/worse on
# high-severity cases? Useful context for the final report's trade-off discussion.
strat = df_test.groupby("Status").agg(
    n=("Doctor", "count"),
    rougeL=("rougeL", "mean"),
    perplexity=("perplexity", "mean"),
).round(4)
strat


In [ ]:
# Save everything needed to compare against the fine-tuned model later
os.makedirs("/kaggle/working/results", exist_ok=True)

with open("/kaggle/working/results/baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

df_test.to_csv("/kaggle/working/results/baseline_predictions.csv", index=False)

print("Saved:")
print(" - /kaggle/working/results/baseline_summary.json")
print(" - /kaggle/working/results/baseline_predictions.csv")


In [ ]:
# Eyeball a few predictions vs references
for i in df_test.sample(3, random_state=SEED).index:
    row = df_test.loc[i]
    print("="*100)
    print("SEVERITY:", row["Status"])
    print("PATIENT :", row["Patient"][:300], "...")
    print("-"*100)
    print("REFERENCE DOCTOR:", row["Doctor"][:400])
    print("-"*100)
    print("MODEL PREDICTION:", row["prediction"][:400])


## 10. Next steps

With the baseline numbers saved to `results/baseline_summary.json` and per-example predictions
in `results/baseline_predictions.csv`, the next notebook will:

1. Build the LoRA config (`target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]`,
   `r=16, lora_alpha=32, lora_dropout=0.05`) — attention + MLP, no tokenizer changes.
2. Fine-tune on `train.json` / validate on `val.json`, seed=8 throughout for consistency.
3. Re-run this exact same evaluation pipeline (ROUGE-L, BLEU, PPL, latency) on the LoRA-tuned
   model and diff against `baseline_summary.json`.
4. Compare Prompt Tuning / LoRA / QLoRA per the project spec's three-way comparison, once the
   LoRA baseline above is solid.
